In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict, cross_val_score
from typing import Tuple
from pathlib import Path
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
import yaml

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Функции для оценки моделей

In [3]:
def nasa_scoring_function(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Вычисляет метрику NASA Scoring Function."""
    errors = y_pred - y_true
    scores = np.zeros_like(errors, dtype=float)
    
    late_mask = errors < 0
    scores[late_mask] = np.exp(-errors[late_mask] / 13) - 1
    
    early_mask = errors >= 0
    scores[early_mask] = np.exp(errors[early_mask] / 10) - 1
    
    return np.mean(scores)

def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, ...]:
    """Вычисляет все метрики для оценки модели."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    nasa_score = nasa_scoring_function(y_true, y_pred)
    
    metrics = {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'NASA_Score': nasa_score
    }
    
    return metrics

In [4]:
def evaluate_with_cv(X: pd.DataFrame, y: pd.Series, model, cv_folds: int = 5) -> dict:
    """Оценивает модель с помощью кросс-валидации."""
    
    y_pred = cross_val_predict(model, X, y, cv=cv_folds)
    metrics = evaluate_model(y, y_pred)
    
    return metrics

## Baseline для каждого датасета

В качестве baseline'ов будут выступать модели CatBoost и Random Forest без подобранных гиперпараметров

In [43]:
def run_baselines(X_train: pd.DataFrame, y_train: pd.Series, dataset: str = "FD001", 
                  cv_folds: int = 5, random_state: int = 0):
    print(f'Запуск baseline для датасета: {dataset}...')
    y_array = np.array(y_train).ravel()
    results = {
        'dataset': dataset,
        'cv_results': {}
    }

    print(f'Random Forest baseline...')
    rf_model = RandomForestRegressor(random_state=random_state, n_jobs=-1)
    rf_cv_metrics = evaluate_with_cv(X_train, y_array, rf_model)
    
    results['cv_results']['RandomForest'] = rf_cv_metrics

    print(f'Catboost baseline...')
    cat_model = CatBoostRegressor(random_seed=random_state, verbose=False, allow_writing_files=False)
    cat_cv_metrics = evaluate_with_cv(X_train, y_array, cat_model)

    results['cv_results']['CatBoost'] = cat_cv_metrics

    print(f'Результаты baseline:')
    
    comparison_df = pd.DataFrame({
        'Model': ['Random Forest', 'CatBoost'],
        'MAE': [
            results['cv_results']['RandomForest']['MAE'],
            results['cv_results']['CatBoost']['MAE']
        ],
        'RMSE': [
            results['cv_results']['RandomForest']['RMSE'],
            results['cv_results']['CatBoost']['RMSE']
        ],
        'R2': [
            results['cv_results']['RandomForest']['R2'],
            results['cv_results']['CatBoost']['R2']
        ],
        'NASA_Score': [
            results['cv_results']['RandomForest']['NASA_Score'],
            results['cv_results']['CatBoost']['NASA_Score']
        ]
    })
    
    print("\n" + comparison_df.to_string(index=False))
    print()
    
    return comparison_df

In [45]:
datasets_dir = Path('../data/processed')
datasets = ['FD001', 'FD002', 'FD003', 'FD004']
all_results = {}

for dataset in datasets:
    X_train_path = datasets_dir / dataset / 'X_train.csv'
    y_train_path = datasets_dir / dataset / 'y_train.csv'
    X_train = pd.read_csv(X_train_path)
    y_train = pd.read_csv(y_train_path)
    
    all_results[dataset] = run_baselines(X_train, y_train, dataset)

Запуск baseline для датасета: FD001...
Random Forest baseline...
Catboost baseline...
Результаты baseline:

        Model       MAE      RMSE       R2  NASA_Score
Random Forest 16.175563 21.251006 0.759649   15.545013
     CatBoost 14.778910 19.417625 0.799331    9.603377

Запуск baseline для датасета: FD002...
Random Forest baseline...
Catboost baseline...
Результаты baseline:

        Model       MAE      RMSE       R2  NASA_Score
Random Forest 18.516681 24.245264 0.687173   28.208057
     CatBoost 18.079065 23.552377 0.704798   26.455297

Запуск baseline для датасета: FD003...
Random Forest baseline...
Catboost baseline...
Результаты baseline:

        Model       MAE      RMSE       R2  NASA_Score
Random Forest 11.918813 18.219854 0.822762    9.962920
     CatBoost 11.629271 17.075041 0.844335    7.063821

Запуск baseline для датасета: FD004...
Random Forest baseline...
Catboost baseline...
Результаты baseline:

        Model       MAE      RMSE       R2  NASA_Score
Random Forest 1

## Hyperparameters tune 

In [4]:
def objective_rf(trial, X_train, y_train, cv_folds=3):
    """Целевая функция для Random Forest"""
    # Пространство гиперпараметров
    n_estimators = trial.suggest_int('n_estimators', 50, 500, step=50)
    max_depth = trial.suggest_int('max_depth', 5, 15)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 8)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    
    # Модель
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=0,
        n_jobs=-1
    )
    
    # Оценка на кросc-вал.
    scores = cross_val_score(model, X_train, y_train, cv=cv_folds, 
                            scoring='neg_root_mean_squared_error')
    rmse = -np.mean(scores)
    
    return rmse

def objective_catboost(trial, X_train, y_train, cv_folds=3):
    """Целевая функция для CatBoost."""
    # Пространство гиперпараметров
    iterations = trial.suggest_int('iterations', 100, 800, step=50)
    depth = trial.suggest_int('depth', 1, 7)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True)
    border_count = trial.suggest_int('border_count', 32, 240, step=16)
    
    # Модель
    model = CatBoostRegressor(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        border_count=border_count,
        random_seed=0,
        verbose=False,
        allow_writing_files=False
    )
    
    # Кросс-валидация
    scores = cross_val_score(model, X_train, y_train, cv=cv_folds,
                            scoring='neg_root_mean_squared_error')
    rmse = -np.mean(scores)
    
    return rmse

In [5]:
def run_optimization_rf(X_train, y_train, dataset_name="FD001", n_trials=100, cv_folds=3):
    """Запускает оптимизацию гиперпараметров для Random Forest."""
    print()
    print(f"Оптимизация гиперпараметров Random Forest: {dataset_name}")
    
    study = optuna.create_study(direction='minimize', study_name=f'rf_{dataset_name}')
    
    study.optimize(
        lambda trial: objective_rf(trial, X_train, y_train, cv_folds),
        n_trials=n_trials,
        show_progress_bar=True
    )
    
    print(f"\nЛучшие параметры:")
    for key, value in study.best_params.items():
        print(f"   {key}: {value}")
    print(f"   Лучший RMSE: {study.best_value:.4f}")
    
    return study


def run_optimization_catboost(X_train, y_train, dataset_name="FD001", n_trials=100, cv_folds=3):
    """Запускает оптимизацию гиперпараметров для CatBoost."""
    print()
    print(f"Оптимизация гиперпараметров CatBoost: {dataset_name}")
    
    study = optuna.create_study(direction='minimize', study_name=f'catboost_{dataset_name}')
    
    study.optimize(
        lambda trial: objective_catboost(trial, X_train, y_train, cv_folds),
        n_trials=n_trials,
        show_progress_bar=True
    )
    
    print(f"\nЛучшие параметры:")
    for key, value in study.best_params.items():
        print(f"   {key}: {value}")
    print(f"   Лучший RMSE: {study.best_value:.4f}")
    
    return study

In [6]:
def train_best_model_rf(X_train, y_train, best_params):
    """Обучает Random Forest с лучшими параметрами (заданными) на всех данных."""
    model = RandomForestRegressor(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        min_samples_split=best_params['min_samples_split'],
        min_samples_leaf=best_params['min_samples_leaf'],
        max_features=best_params['max_features'],
        random_state=0,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    return model


def train_best_model_catboost(X_train, y_train, best_params):
    """Обучает CatBoost с лучшими параметрами (заданными) на всех данных."""
    model = CatBoostRegressor(
        iterations=best_params['iterations'],
        depth=best_params['depth'],
        learning_rate=best_params['learning_rate'],
        l2_leaf_reg=best_params['l2_leaf_reg'],
        border_count=best_params['border_count'],
        random_seed=0,
        verbose=False,
        allow_writing_files=False
    )
    model.fit(X_train, y_train)
    return model

In [8]:
datasets_dir = Path('../data/processed')
datasets = ['FD001', 'FD002', 'FD003', 'FD004']
all_studies = {}
n_trials = 50

for dataset in datasets:
    X_train = pd.read_csv(Path(datasets_dir) / dataset / 'X_train.csv')
    y_train = pd.read_csv(Path(datasets_dir) / dataset / 'y_train.csv').squeeze()
    
    study_rf = run_optimization_rf(X_train, y_train, dataset, n_trials)
    study_cat = run_optimization_catboost(X_train, y_train, dataset, n_trials)
        
    all_studies[dataset] = {
        'rf': study_rf,
        'catboost': study_cat
    }

    print(f"\nСравнение для {dataset}:")
    print(f"   RF: RMSE optimized = {study_rf.best_value:.4f}")
    print(f"   CatBoost: RMSE optimized = {study_cat.best_value:.4f}")


Оптимизация гиперпараметров Random Forest: FD001


  0%|          | 0/50 [00:00<?, ?it/s]


Лучшие параметры:
   n_estimators: 200
   max_depth: 13
   min_samples_split: 3
   min_samples_leaf: 6
   max_features: log2
   Лучший RMSE: 19.7160

Оптимизация гиперпараметров CatBoost: FD001


  0%|          | 0/50 [00:00<?, ?it/s]


Лучшие параметры:
   iterations: 750
   depth: 7
   learning_rate: 0.010008594974926458
   l2_leaf_reg: 2.789428705259572e-08
   border_count: 48
   Лучший RMSE: 18.4758

Сравнение для FD001:
   RF: RMSE optimized = 19.7160
   CatBoost: RMSE optimized = 18.4758

Оптимизация гиперпараметров Random Forest: FD002


...


Лучшие параметры:
   iterations: 450
   depth: 7
   learning_rate: 0.01449940790972958
   l2_leaf_reg: 2.264973118922654
   border_count: 48
   Лучший RMSE: 16.1371

Сравнение для FD003:
   RF: RMSE optimized = 17.1017
   CatBoost: RMSE optimized = 16.1371

Оптимизация гиперпараметров Random Forest: FD004


  0%|          | 0/50 [00:00<?, ?it/s]


Лучшие параметры:
   n_estimators: 500
   max_depth: 13
   min_samples_split: 4
   min_samples_leaf: 1
   max_features: sqrt
   Лучший RMSE: 21.2876

Оптимизация гиперпараметров CatBoost: FD004


  0%|          | 0/50 [00:00<?, ?it/s]


Лучшие параметры:
   iterations: 550
   depth: 6
   learning_rate: 0.02695189375656805
   l2_leaf_reg: 4.752167058010921e-06
   border_count: 64
   Лучший RMSE: 21.1594

Сравнение для FD004:
   RF: RMSE optimized = 21.2876
   CatBoost: RMSE optimized = 21.1594


In [5]:
best_params = {
    'FD001': {
        'rf': {
            'n_estimators': 200,
            'max_depth': 13,
            'min_samples_split': 3,
            'min_samples_leaf': 6,
            'max_features': 'log2'
        },
        'catboost': {
            'iterations': 750,
            'depth': 7,
            'learning_rate': 0.010008594974926458,
            'l2_leaf_reg': 2.789428705259572e-08,
            'border_count': 48
        }
    },
    'FD002': {
        'rf': {
            'n_estimators': 350,
            'max_depth': 12,
            'min_samples_split': 3,
            'min_samples_leaf': 5,
            'max_features': 'sqrt'
        },
        'catboost': {
            'iterations': 600,
            'depth': 7,
            'learning_rate': 0.012257154926226671,
            'l2_leaf_reg': 0.007540339892874158,
            'border_count': 48
        }
    },
    'FD003': {
        'rf': {
            'n_estimators': 450,
            'max_depth': 12,
            'min_samples_split': 9,
            'min_samples_leaf': 8,
            'max_features': 'sqrt'
        },
        'catboost': {
            'iterations': 450,
            'depth': 7,
            'learning_rate': 0.01449940790972958,
            'l2_leaf_reg': 2.264973118922654,
            'border_count': 48
        }
    },
    'FD004': {
        'rf': {
            'n_estimators': 500,
            'max_depth': 13,
            'min_samples_split': 4,
            'min_samples_leaf': 1,
            'max_features': 'sqrt'
        },
        'catboost': {
            'iterations': 550,
            'depth': 6,
            'learning_rate': 0.02695189375656805,
            'l2_leaf_reg': 4.752167058010921e-06,
            'border_count': 64
        }
    }
}

def evaluate_on_test(dataset, best_params_rf, best_params_cat):
    """Оценивает лучшие модели на тестовых данных для указанного датасета."""
    print(f"\nОценка на тесте для датасета {dataset}")
    
    X_train = pd.read_csv(Path('../data/processed') / dataset / 'X_train.csv')
    y_train = pd.read_csv(Path('../data/processed') / dataset / 'y_train.csv').squeeze()
    
    X_test = pd.read_csv(Path('../data/processed') / dataset / 'X_test.csv')
    y_test = pd.read_csv(Path('../data/processed') / dataset / 'y_test.csv').squeeze()
    
    print("\nОбучение моделей...")
    
    rf_model = RandomForestRegressor(
        n_estimators=best_params_rf['n_estimators'],
        max_depth=best_params_rf['max_depth'],
        min_samples_split=best_params_rf['min_samples_split'],
        min_samples_leaf=best_params_rf['min_samples_leaf'],
        max_features=best_params_rf['max_features'],
        random_state=0,
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    
    cat_model = CatBoostRegressor(
        iterations=best_params_cat['iterations'],
        depth=best_params_cat['depth'],
        learning_rate=best_params_cat['learning_rate'],
        l2_leaf_reg=best_params_cat['l2_leaf_reg'],
        border_count=best_params_cat['border_count'],
        random_seed=0,
        verbose=False,
        allow_writing_files=False
    )
    cat_model.fit(X_train, y_train)
    
    
    y_pred_rf = rf_model.predict(X_test)
    y_pred_cat = cat_model.predict(X_test)
    
    results = {}
    if y_test is not None:
        metrics_rf = evaluate_model(y_test, y_pred_rf)
        metrics_cat = evaluate_model(y_test, y_pred_cat)
        
        print(f"\nRandom Forest на тесте:")
        for metric, value in metrics_rf.items():
            print(f"   {metric}: {value:.4f}")
        
        print(f"\nCatBoost на тесте:")
        for metric, value in metrics_cat.items():
            print(f"   {metric}: {value:.4f}")
        
        results = {
            'rf': metrics_rf,
            'catboost': metrics_cat
        }
    else:
        print("\nНет тестовых RUL для оценки")
    
    return results, rf_model, cat_model

In [6]:
test_results = {}
for dataset in ['FD001', 'FD002', 'FD003', 'FD004']:
    results, _, _ = evaluate_on_test(
        dataset,
        best_params[dataset]['rf'],
        best_params[dataset]['catboost']
    )
    test_results[dataset] = results


Оценка на тесте для датасета FD001

Обучение моделей...

Random Forest на тесте:
   MAE: 12.6252
   RMSE: 17.0971
   R2: 0.8234
   NASA_Score: 6.9406

CatBoost на тесте:
   MAE: 11.8018
   RMSE: 15.9861
   R2: 0.8456
   NASA_Score: 5.6294

Оценка на тесте для датасета FD002

Обучение моделей...

Random Forest на тесте:
   MAE: 16.2507
   RMSE: 20.6051
   R2: 0.7831
   NASA_Score: 11.0632

CatBoost на тесте:
   MAE: 15.8639
   RMSE: 20.3709
   R2: 0.7880
   NASA_Score: 10.9092

Оценка на тесте для датасета FD003

Обучение моделей...

Random Forest на тесте:
   MAE: 12.2317
   RMSE: 17.4880
   R2: 0.8095
   NASA_Score: 14.2417

CatBoost на тесте:
   MAE: 12.6798
   RMSE: 17.5609
   R2: 0.8079
   NASA_Score: 13.5024

Оценка на тесте для датасета FD004

Обучение моделей...

Random Forest на тесте:
   MAE: 16.3731
   RMSE: 21.4300
   R2: 0.7670
   NASA_Score: 20.8985

CatBoost на тесте:
   MAE: 15.4826
   RMSE: 20.6420
   R2: 0.7839
   NASA_Score: 17.9669


In [14]:
# Сохраним параметры в конфиг
config_dir = Path('../config')
config_dir.mkdir(parents=True, exist_ok=True)

yaml_path = config_dir / 'best_params.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(best_params, f, default_flow_style=False, allow_unicode=True)
print(f'YAML сохранен: {yaml_path}')

YAML сохранен: ..\config\best_params.yaml
